<a href="https://colab.research.google.com/github/lathikasuriyamoorthy-29/trendpulse-lathikasuriyamoorthy/blob/main/Data_Collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import requests
import time
import json
import os
import re
from datetime import datetime


# --------------------------------------------------
# 1. API URLs
# --------------------------------------------------

TOP_STORIES_URL = "https://hacker-news.firebaseio.com/v0/topstories.json"
ITEMS_URL = "https://hacker-news.firebaseio.com/v0/item/{}.json"


# --------------------------------------------------
# 2. Categories and keywords
# --------------------------------------------------

categories = categories = {
    "sports": [
        "NFL", "NBA", "MLB", "NHL", "FIFA", "UEFA", "Olympics", "Olympic",
        "sport", "sports", "soccer", "football", "basketball", "baseball",
        "cricket", "tennis", "golf", "rugby", "hockey", "cycling", "marathon",
        "athlete", "athletes", "player", "players", "league", "championship",
        "tournament", "coach", "stadium", "Formula 1", "F1", "World Cup",
        "Premier League", "Wimbledon", "boxing", "UFC", "chess"
    ],
    "science": [
        "research", "researchers", "study", "studies", "scientist", "scientists",
        "physics", "biology", "chemistry", "astronomy", "NASA", "ESA", "genome",
        "DNA", "discovery", "space", "telescope", "planet", "planets", "Mars",
        "Moon", "quantum", "vaccine", "species", "fossil", "brain", "cancer",
        "medicine", "neuroscience", "experiment", "universe", "asteroid"
    ],
    "entertainment": [
        "movie", "movies", "film", "films", "music", "song", "songs", "album",
        "Netflix", "Disney", "game", "games", "gaming", "book", "books",
        "novel", "award", "awards", "streaming", "series", "anime", "podcast",
        "concert", "actor", "actress", "director", "Oscar", "Spotify",
        "YouTube", "TV", "comedy", "theater", "theatre"
    ],
    "worldnews": [
        "war", "government", "country", "countries", "president", "election",
        "elections", "climate", "global", "Ukraine", "Russia", "China", "India",
        "Israel", "Gaza", "Iran", "EU", "UN", "NATO", "tariff", "tariffs",
        "sanctions", "minister", "parliament", "court", "law", "police",
        "protest", "protests", "economy", "inflation", "immigration",
        "military", "treaty", "senate", "congress"
    ],
    "technology": [
        "AI", "software", "tech", "code", "computer", "data", "cloud", "API",
        "GPU", "LLM", "programming", "developer", "developers", "open source",
        "Linux", "Python", "Rust", "JavaScript", "database", "startup",
        "security", "browser", "app", "apps", "hardware", "chip", "server",
        "algorithm", "model", "robot", "Apple", "Google", "Microsoft"
    ]
}


# --------------------------------------------------
# 3. Function to classify a story
# --------------------------------------------------

def get_category(title):

    for category, keywords in categories.items():

        for keyword in keywords:

            if re.search(r"\b"+re.escape(keyword)+r"\b",title,re.IGNORECASE):
                return category

    return None


# --------------------------------------------------
# 4. Get the first 500 top-story IDs
# --------------------------------------------------

response = requests.get(TOP_STORIES_URL)

story_ids = response.json()[:500]

print("Story IDs fetched:", len(story_ids))


# --------------------------------------------------
# 5. Storage
# --------------------------------------------------

collected_stories = []

category_counts = {
    "technology": 0,
    "worldnews": 0,
    "sports": 0,
    "science": 0,
    "entertainment": 0
}


# --------------------------------------------------
# 6. Collect stories category by category
# --------------------------------------------------

for category in categories:

    print("\nCollecting:", category)

    for story_id in story_ids:

        # Stop when this category reaches 25
        if category_counts[category] >= 25:
            break

        # Create URL for this particular story
        story_url = ITEMS_URL.format(story_id)

        try:
            story_response = requests.get(
                story_url,
                timeout=10
            )

            story = story_response.json()

        except requests.RequestException:
            continue

        # to make sure we received a story
        if not story or story.get("type") != "story":
            continue

        # Get title
        title = story.get("title", "")

        # Determine category
        detected_category = get_category(title)

        # Skip if it doesn't belong to the category
        # currently being collected
        if detected_category != category:
            continue

        # to create our own structured dictionary for the story informations
        story_data = {
            "post_id": story.get("id"),
            "title": title,
            "category": category,
            "score": story.get("score", 0),
            "num_comments": story.get("descendants", 0),
            "author": story.get("by"),
            "collected_at": datetime.now().isoformat()
        }

        # Store the story
        collected_stories.append(story_data)

        # Increase category count
        category_counts[category] += 1

    # Wait 2 seconds before starting next category
    time.sleep(2)


# --------------------------------------------------
# 7. Display results
# --------------------------------------------------

print("\n-----------------------------")
print("COLLECTION COMPLETE")
print("-----------------------------")

print("Total stories:", len(collected_stories))

print("\nCategory counts:")

for category, count in category_counts.items():
    print(category, ":", count)


# --------------------------------------------------
# 8. Save as JSON
# --------------------------------------------------

os.makedirs("data", exist_ok=True)

filename = (
    "data/trends_"
    + datetime.now().strftime("%Y%m%d")
    + ".json"
)

with open(filename, "w", encoding="utf-8") as file:

    json.dump(
        collected_stories,
        file,
        indent=4,
        ensure_ascii=False
    )

print("\nJSON file saved as:")
print(filename)

Story IDs fetched: 500

Collecting: sports

Collecting: science

Collecting: entertainment

Collecting: worldnews

Collecting: technology

-----------------------------
COLLECTION COMPLETE
-----------------------------
Total stories: 89

Category counts:
technology : 25
worldnews : 22
sports : 3
science : 25
entertainment : 14

JSON file saved as:
data/trends_20261005.json


In [4]:
import pandas as pd
df = pd.read_json("data/trends_20261005.json")
print(df[["title","category"]].sample(20,random_state = 1).to_string())

                                                                            title       category
60                            How Singapore's government-run dating service works      worldnews
70                     How to scale intent, quality, and artistry with AI [video]     technology
53                                              The Escalation of War in Ethiopia      worldnews
54     Plague death triggers quarantine, mask mandates in Russia's Irkutsk region      worldnews
77           The work by Valve's Timur Kristóf on improving old AMD GPUs on Linux     technology
73             Kagi Ends Orion Browser Development for Linux, Will Open-Source It     technology
59                                  Is Russia using AI for disinformation in CAR?      worldnews
41                                  The Rise of the Meritocracy (satirical novel)  entertainment
45                  Three AI agents, two countries, and one uneven world wide web      worldnews
38                            